# Stage 2 — Cosine Similarity Recommendation
**[STUDENT VERSION — fill in the blanks]**

Stage này trả lời câu hỏi: **“Với sở thích hiện tại, điểm đến nào phù hợp nhất?”**

Mỗi điểm đến và mỗi profile người dùng được biểu diễn trong cùng một không gian 13 chiều. Mười hai chiều đầu là các feature cố định; chiều thứ 13 là `month_score`, được tính tại thời điểm truy vấn vì nó phụ thuộc vào tháng người dùng muốn đi.

### Sau stage này, bạn có thể
- phân biệt content-based filtering với collaborative filtering;
- dựng hai vector đúng thứ tự feature và tự tính cosine similarity;
- giải thích vì sao `month_score` phải được tạo lúc query;
- score toàn bộ điểm đến, sắp hạng và lấy top-K;
- phân biệt một sanity check với đánh giá recommender có ground truth.

### Luồng xử lý
```text
user profile + travel_month
            │
            ▼
   user vector U (13D)  ── so khớp ──  destination vector Dᵢ (13D)
            │                               ▲
            └──────── cosine(U, Dᵢ) ─────────┘
                            │
                            ▼
                    sort giảm dần → top-K
```

- Input: `../stage1/stage1_dataset.csv` — full 13-feature dataset from Stage 1.
- Input: `stage2_user_profiles.csv` — test user preference profiles.
- Output: `stage2_results.csv` — top-K recommendations per profile.
- Output: `stage2_results.json` — same in JSON format.

**Algorithm:** Content-based filtering using Cosine Similarity in ℝ¹³.

$$\text{sim}(\mathbf{U}, \mathbf{D}_i) = \frac{\mathbf{U} \cdot \mathbf{D}_i}{\|\mathbf{U}\| \cdot \|\mathbf{D}_i\|}$$

The 13th dimension (`month_score`) is encoded **on-the-fly** per query:
- `1.0` — travel month matches destination's best months  
- `0.5` — user không chỉ định tháng; đây là mức trung gian theo quy ước  
- `0.0` — tháng không nằm trong mùa phù hợp theo dữ liệu

> ⚠️ `month_score` là một tín hiệu ngữ cảnh nằm bên trong cosine, **không phải** bộ lọc mùa cứng. Giá trị `0.0` không bảo đảm điểm cuối luôn giảm; phần TODO 1 sẽ giải thích kỹ hơn.

> 💡 Cells marked `# TODO` require you to fill in the code.

## 1. Đây là loại recommender nào?

Notebook dùng **content-based filtering**: hệ thống so sở thích do người dùng khai báo với metadata của từng điểm đến. `stage2_user_profiles.csv` là tập truy vấn thử, không phải lịch sử tương tác để huấn luyện model.

| Cách tiếp cận | Dữ liệu cần | Có trong notebook? |
|---|---|---|
| Content-based | feature của item + sở thích user | Có |
| Collaborative filtering | rating/click/booking của nhiều user | Không |
| Hybrid | kết hợp hai nguồn trên | Không |

**Ưu điểm:** dễ giải thích vì có thể chỉ ra feature nào khớp; một điểm đến mới vẫn có thể được gợi ý nếu đã có đủ metadata.  
**Giới hạn:** kết quả phụ thuộc mạnh vào feature do con người thiết kế, không tự học từ hành vi và dễ chỉ đề xuất những lựa chọn giống sở thích đã khai báo.


In [19]:
import csv, json, math
from pathlib import Path
import numpy as np
import pandas as pd


In [20]:
BASE_DIR     = Path('.')
DATASET_CSV  = BASE_DIR / '../input/stage1/stage1_dataset.csv'
PROFILES_CSV = BASE_DIR / '../input/stage2/stage2_user_profiles.csv'
RESULTS_CSV  = BASE_DIR / '../input/stage2/stage2_results.csv'
RESULTS_JSON = BASE_DIR / '../input/stage2/stage2_results.json'

def read_csv(path):
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

dataset  = read_csv(DATASET_CSV)
profiles = read_csv(PROFILES_CSV)
print(profiles)
print(f'Destinations: {len(dataset)}')
print(f'Profiles: {len(profiles)}')

[{'profile_name': 'user_1', 'beach': '1.0', 'history': '0.3', 'food': '0.7', 'nature': '0.6', 'adventure': '0.2', 'culture': '0.4', 'relax': '1.0', 'photo': '0.6', 'budget': '0.7', 'family': '1.0', 'crowd': '0.6', 'month': 'all-year', 'top_k': '10'}, {'profile_name': 'user_2', 'beach': '0.3', 'history': '0.2', 'food': '0.2', 'nature': '1.0', 'adventure': '1.0', 'culture': '0.2', 'relax': '0.1', 'photo': '0.8', 'budget': '0.2', 'family': '0.0', 'crowd': '0.1', 'month': 'all-year', 'top_k': '10'}, {'profile_name': 'user_3', 'beach': '0.1', 'history': '1.0', 'food': '0.3', 'nature': '0.4', 'adventure': '0.2', 'culture': '0.9', 'relax': '0.5', 'photo': '0.7', 'budget': '0.7', 'family': '0.6', 'crowd': '0.4', 'month': 'all-year', 'top_k': '10'}, {'profile_name': 'user_4', 'beach': '0.3', 'history': '0.4', 'food': '1.0', 'nature': '0.3', 'adventure': '0.1', 'culture': '0.9', 'relax': '0.4', 'photo': '0.5', 'budget': '0.8', 'family': '0.5', 'crowd': '0.9', 'month': 'all-year', 'top_k': '10'},

## 🔧 TODO 1 — `encode_month()`

`best_months` được giữ dạng chuỗi ở Stage 1 vì chưa biết người dùng sẽ đi tháng nào. Tại Stage 2, `encode_month()` mới kết hợp chuỗi này với `travel_month` để tạo feature theo từng query. Cách trì hoãn việc mã hóa đến khi có đủ ngữ cảnh thường được gọi là **late binding**.

Ví dụ: `best_months='mar-apr-may'` cho kết quả `1.0` với `travel_month='apr'`, nhưng cho `0.0` với `travel_month='oct'`. Cùng một điểm đến vì thế có thể nhận hai vector khác nhau ở hai truy vấn.

| Trường hợp | Return |
|---|---|
| `travel_month` rỗng / None | `0.5` — mức trung gian theo quy ước |
| `best_months == 'all-year'` | `1.0` — điểm đẹp quanh năm |
| `travel_month` nằm trong `best_months` | `1.0` — đúng mùa |
| Còn lại | `0.0` — không khớp mùa trong dữ liệu |

> ⚠️ Hàm này được gọi **bên trong** `cosine_sim_13()` mỗi lần tính score.  
> Không pre-compute vì giá trị phụ thuộc vào `travel_month` của user — chỉ biết lúc query.

### Một giới hạn dễ bỏ sót

Gắn mùa vào cosine là cách đơn giản để thêm ngữ cảnh, nhưng `0.0` **không phải một phép trừ điểm trực tiếp**. Nó đồng thời làm thay đổi tử số và độ dài vector điểm đến, nên không bảo đảm thứ hạng sẽ luôn thấp hơn trường hợp `1.0`. Tương tự, nhãn `neutral` cho `0.5` chỉ là tên quy ước; chiều này vẫn tham gia phép tính.

Nếu yêu cầu nghiệp vụ là “trái mùa chắc chắn bị loại/phạt”, hướng mở rộng phù hợp hơn là lọc trước khi xếp hạng hoặc nhân base cosine với một `season_factor` ở bước sau. Bài thực hành này giữ cách 13D để minh họa **contextual feature**.

> **Data contract:** chuỗi tháng dùng mã `jan`, `feb`, ..., `dec`, phân tách bằng dấu `-`. Viết sai mã hoặc thừa khoảng trắng sẽ làm phép so khớp thất bại.


In [21]:
ALL_FEATURES = [
    'beach', 'history', 'food', 'nature', 'adventure',
    'culture', 'relax', 'photo',
    'budget', 'family', 'crowd',
    'month_score',
]

def encode_month(best_months: str, travel_month: str) -> float:
    """
    Returns:
        1.0  nếu travel_month khớp với best_months (hoặc all-year)
        0.5  nếu travel_month rỗng (neutral)
        0.0  nếu trái mùa
    """
    # TODO 1: Điền logic vào đây
    if not travel_month:
        return 0.5  # ← neutral

    if best_months == 'all-year':
        return 1.0  # ← luôn match

    if travel_month in best_months.split('-'):
        return 1.0  # ← đúng mùa

    return 0.0  # ← trái mùa


# Test
print(encode_month('mar-apr-may-jun-jul-aug', 'apr'))  # → 1.0
print(encode_month('mar-apr-may-jun-jul-aug', 'oct'))  # → 0.0
print(encode_month('mar-apr-may-jun-jul-aug', ''))     # → 0.5
print(encode_month('all-year', 'oct'))                 # → 1.0


1.0
0.0
0.5
1.0


## 🔧 TODO 2 — `cosine_sim_13()`

Hàm này là lõi của recommender. Ta dựng hai vector **cùng số chiều và cùng thứ tự feature**:
- vector user `U`: mức mong muốn cho từng feature;
- vector điểm đến `D`: mức độ điểm đến có feature đó.

Sau đó tính cosine similarity trong ℝ¹³ giữa hai vector đó.
`month_score` được gắn động ngay khi tính điểm, vì giá trị này phụ thuộc vào tháng user nhập ở từng lần query.

**Công thức:**
```
sim = dot(U, D) / (norm(U) × norm(D))
    = Σ(u_k × d_k) / (√(Σu_k²) × √(Σd_k²))
```

### Ví dụ 2 chiều

Giả sử user chỉ quan tâm feature thứ nhất: `U = (1, 0)`.
- Điểm A: `D_A = (0.8, 0.2)` → cosine ≈ `0.970`.
- Điểm B: `D_B = (0.8, 0.8)` → cosine ≈ `0.707`.

Hai điểm cùng có `0.8` ở feature user muốn, nhưng A có **hướng vector** gần U hơn nên xếp cao hơn. Đây là lý do cosine đo “mẫu sở thích tương đối”, không chỉ cộng các feature mạnh.

**Lưu ý quan trọng:**
- `month_score` của destination: gọi `encode_month()` on-the-fly
- `month_score` của user: luôn = **1.0** (user luôn muốn đi đúng mùa)
- Tất cả giá trị ở đây không âm nên score nằm trong `[0, 1]`; cosine tổng quát có thể nằm trong `[-1, 1]`.
- Score là độ giống hướng, **không phải xác suất, confidence hay accuracy**.
- `0` ở user vector nghĩa là không tạo đóng góp dương cho feature đó, không phải một “dislike” có trọng số âm.
- Nếu norm = 0 → return 0.0 để tránh chia cho 0.
- Vì cosine bất biến khi nhân toàn bộ một vector với cùng hằng số dương, **tỷ lệ giữa các chiều** quan trọng hơn độ lớn tuyệt đối.


In [22]:
def cosine_sim_13(user_prefs: dict, place: dict, travel_month: str = None) -> float:
    # Bước 1: Build destination vector (12 features + month on-the-fly)
    d_vec = {f: float(place.get(f, 0.0)) for f in ALL_FEATURES if f != 'month_score'}
    d_vec['month_score'] = encode_month(place['best_months'], travel_month)  # ← encode_month(place['best_months'], travel_month)

    # Bước 2: Build user vector (user luôn có month_score = 1.0)
    u_vec = dict(user_prefs)
    u_vec['month_score'] = 1.0  # ← 1.0

    # Bước 3: Tính dot product và norms
    dot = nu = nv = 0.0
    for f in ALL_FEATURES:
        u = float(u_vec.get(f, 0.0))
        d = float(d_vec.get(f, 0.0))
        # TODO: tích lũy dot, nu, nv
        dot += u * d  # ← u * d
        nu  += u * u  # ← u * u
        nv  += d * d  # ← d * d

    # Bước 4: Guard + return
    if nu == 0 or nv == 0:
        return 0.0
    return round(dot / (math.sqrt(nu) * math.sqrt(nv)), 4)  # ← dot / (math.sqrt(nu) * math.sqrt(nv))


# Test nhanh
test_prefs = {'beach':1.0,'history':0.0,'food':0.5,'nature':0.6,
              'adventure':0.2,'culture':0.1,'relax':0.8,'photo':0.7,
              'budget':0.7,'family':0.6,'access':0.7,'crowd':0.3}
score = cosine_sim_13(test_prefs, dataset[0], travel_month='jun')
print(f'Test score for {dataset[0]["place"]}: {score}')
print('Nếu score trong khoảng [0, 1] là đúng.')


Test score for Phong Nha: 0.7876
Nếu score trong khoảng [0, 1] là đúng.


## 🔧 TODO 3 — Run All Profiles + Rank

Với mỗi profile, ta thực hiện ba bước:
1. tính score cho toàn bộ `n` điểm đến;
2. sort score giảm dần;
3. lấy `top_k` phần tử đầu.

Vì số chiều `d = 13` là cố định, scoring có độ phức tạp `O(n × d)` và sort là `O(n log n)`. Với 38 điểm đến, cách làm duyệt toàn bộ vừa rõ ràng vừa đủ nhanh.

> **Top-K là ranking, không phải classification.** Hệ thống không dự đoán nhãn đúng/sai cho từng điểm; nó tạo một thứ tự ưu tiên tương đối. Do score được làm tròn 4 chữ số trước khi sort, có thể xuất hiện tie; Python giữ thứ tự ban đầu của các phần tử có cùng key.


In [23]:
# TODO 3: Score tất cả destinations cho mỗi profile, sort và lấy top-K
all_results = []
result_rows = []

for prof in profiles:
    name         = prof['profile_name']
    travel_month = prof.get('travel_month') or None
    top_k        = int(prof.get('top_k', 10))
    user_prefs   = {f: float(prof[f]) for f in ALL_FEATURES
                    if f != 'month_score' and f in prof}

    # TODO: score tất cả destinations
    scored = []
    for place in dataset:
        score       = cosine_sim_13(user_prefs, place, travel_month)  # ← cosine_sim_13(user_prefs, place, travel_month)
        month_score = encode_month(place['best_months'], travel_month)  # ← encode_month(place['best_months'], travel_month)
        scored.append({'place': place['place'], 'province': place['province'],
                       'score': score, 'month_score': month_score})

    # TODO: sort giảm dần theo score và lấy top_k
    scored.sort(key=lambda x: x['score'], reverse=True)  # ← key=lambda x: x['score']
    top = scored[:top_k]

    all_results.append({'profile': name, 'travel_month': travel_month,
                        'top_k': top_k, 'results': top})
    for rank, r in enumerate(top, 1):
        result_rows.append({
            'profile': name, 'rank': rank,
            'place': r['place'], 'province': r['province'],
            'cosine_score': r['score'],
            'month_match': 'match' if r['month_score']==1.0
                           else 'neutral' if r['month_score']==0.5
                           else 'off-season',
        })

print(f'Profiles processed: {len(profiles)}')
print(f'Total result rows: {len(result_rows)}')
print(result_rows)
all_results

Profiles processed: 10
Total result rows: 100
[{'profile': 'user_1', 'rank': 1, 'place': 'Đầm Chuồn', 'province': 'Huế', 'cosine_score': 0.9062, 'month_match': 'neutral'}, {'profile': 'user_1', 'rank': 2, 'place': 'Cầu qua Phá Tam Giang', 'province': 'Huế', 'cosine_score': 0.8829, 'month_match': 'neutral'}, {'profile': 'user_1', 'rank': 3, 'place': 'Biển Cửa Việt', 'province': 'Quảng Trị', 'cosine_score': 0.8809, 'month_match': 'neutral'}, {'profile': 'user_1', 'rank': 4, 'place': 'Lăng Cô', 'province': 'Huế', 'cosine_score': 0.8702, 'month_match': 'neutral'}, {'profile': 'user_1', 'rank': 5, 'place': 'Vũng Chùa - Đảo Yến', 'province': 'Quảng Bình', 'cosine_score': 0.8645, 'month_match': 'neutral'}, {'profile': 'user_1', 'rank': 6, 'place': 'Biển Nhật Lệ', 'province': 'Quảng Bình', 'cosine_score': 0.8593, 'month_match': 'neutral'}, {'profile': 'user_1', 'rank': 7, 'place': 'Địa đạo Vịnh Mốc', 'province': 'Quảng Trị', 'cosine_score': 0.8444, 'month_match': 'neutral'}, {'profile': 'user_

[{'profile': 'user_1',
  'travel_month': None,
  'top_k': 10,
  'results': [{'place': 'Đầm Chuồn',
    'province': 'Huế',
    'score': 0.9062,
    'month_score': 0.5},
   {'place': 'Cầu qua Phá Tam Giang',
    'province': 'Huế',
    'score': 0.8829,
    'month_score': 0.5},
   {'place': 'Biển Cửa Việt',
    'province': 'Quảng Trị',
    'score': 0.8809,
    'month_score': 0.5},
   {'place': 'Lăng Cô',
    'province': 'Huế',
    'score': 0.8702,
    'month_score': 0.5},
   {'place': 'Vũng Chùa - Đảo Yến',
    'province': 'Quảng Bình',
    'score': 0.8645,
    'month_score': 0.5},
   {'place': 'Biển Nhật Lệ',
    'province': 'Quảng Bình',
    'score': 0.8593,
    'month_score': 0.5},
   {'place': 'Địa đạo Vịnh Mốc',
    'province': 'Quảng Trị',
    'score': 0.8444,
    'month_score': 0.5},
   {'place': 'Sông Hương',
    'province': 'Huế',
    'score': 0.8442,
    'month_score': 0.5},
   {'place': 'Nhà vườn An Hiên',
    'province': 'Huế',
    'score': 0.8413,
    'month_score': 0.5},
   {

In [24]:
# Chuyển kết quả model thành DataFrame
model_predictions = pd.DataFrame(result_rows)

# Đổi tên cột cho khớp với file reference
model_predictions = model_predictions.rename(
    columns={"profile": "profile_name"}
)

# Chỉ giữ các cột cần thiết để so sánh
model_predictions = model_predictions[
    ["profile_name", "rank", "place"]
]

# Sắp xếp để bảo đảm đúng thứ tự user và rank
model_predictions = model_predictions.sort_values(
    by=["profile_name", "rank"]
).reset_index(drop=True)

# Xuất CSV
model_predictions.to_csv(
    "model_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Đã xuất {len(model_predictions)} kết quả.")
print("File: model_predictions.csv")

display(model_predictions)

Đã xuất 100 kết quả.
File: model_predictions.csv


,profile_name,rank,place
0,user_1,1,Đầm Chuồn
1,user_1,2,Cầu qua Phá Tam Giang
2,user_1,3,Biển Cửa Việt
3,user_1,4,Lăng Cô
4,user_1,5,Vũng Chùa - Đảo Yến
...,...,...,...
95,user_9,6,"Núi Talung, núi Klu"
96,user_9,7,Đền thờ Liễu Hạnh
97,user_9,8,Núi Ngự Bình
98,user_9,9,Lăng Minh Mạng


In [25]:
# Save
headers = ['profile','rank','place','province','cosine_score','month_match']
with RESULTS_CSV.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader(); writer.writerows(result_rows)
with RESULTS_JSON.open('w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print(f'Saved: {RESULTS_CSV.name}')
print(f'Saved: {RESULTS_JSON.name}')


Saved: stage2_results.csv
Saved: stage2_results.json


## Đánh giá kết quả: sanity check khác metric

Các test hiện tại chỉ kiểm tra hàm chạy được, score có đúng range và top-K nhìn có hợp lý hay không. Đó là **sanity check**, chưa chứng minh recommender chính xác vì notebook không có ground truth về những điểm mỗi user thật sự thích/đã chọn.

- `cosine_score = 0.9` không có nghĩa là “90% chính xác”.
- `month_match` là thông tin ngữ cảnh, không phải metric đánh giá.
- Khi có nhãn relevant, có thể dùng `Precision@K`, `Recall@K` và `NDCG@K`.
- Với bài toán du lịch, nên theo dõi thêm coverage, diversity và tỷ lệ kết quả phù hợp mùa.
- Một đánh giá tốt cần so với baseline đơn giản như random, popularity hoặc season-only.

## Tự kiểm tra trước khi sang Stage 3

1. Vì sao user vector và destination vector phải dùng đúng cùng thứ tự feature?
2. Vì sao `best_months` không nên được mã hóa cố định ở Stage 1?
3. `cosine_score` cao có phải là xác suất user sẽ thích điểm đến không?
4. Khi nào nên dùng hard filter theo mùa thay cho `month_score` trong cosine?
5. Nếu tất cả preference bằng 0, guard trong hàm cosine ngăn lỗi gì?
